priner agent installation and configuration

In [1]:
import langchain
print(f"LangChain version: {langchain.__version__}")

# Also check what's available
from langchain import agents
print([x for x in dir(agents) if "create" in x.lower()])

LangChain version: 1.2.0


c:\Users\Admin\Desktop\PrinterAgent\venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
c:\Users\Admin\Desktop\PrinterAgent\venv\Lib\site-packages\langgraph\checkpoint\base\__init__.py:17: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


['create_agent']


In [2]:
# Cell 1: Setup
from dotenv import load_dotenv
load_dotenv()

from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain.agents import create_agent

print("✅ All imports loaded!")

✅ All imports loaded!


In [3]:
# Cell 2: LLM — Ollama (Free, Local)
model = ChatOllama(
    model="llama3.2:3b",  # Change to whatever model you pulled
    base_url="http://localhost:11434",
    temperature=0.0,
)

# Quick test
response = model.invoke("Say 'LLM Ready!' and nothing else.")
print(f"✅ Ollama working: {response.content}")

✅ Ollama working: LLM Ready!


In [4]:
# Cell 3: Tools
@tool
def find_printer(location: str) -> str:
    """Find the Sharp printer for a given office location at the refinery."""
    printers = {
        "GVP": {"name": "GVP", "ip": "172.16.16.47", "model": "BP-50C36"},
        "president": {"name": "president_office", "ip": "172.16.16.46", "model": "BP-50C36"},
        "procurement": {"name": "procurement_office", "ip": "172.16.17.70", "model": "BP-50C36"},
    }
    for key, printer in printers.items():
        if location.lower() in key:
            return f"Found: {printer['name']} | IP: {printer['ip']} | Model: {printer['model']}"
    return f"No printer found for '{location}'. Available: {list(printers.keys())}"


@tool
def register_user(printer_ip: str, user_name: str, user_code: str) -> str:
    """Register a user on the Sharp printer web interface."""
    return f"[SIMULATED] User '{user_name}' registered on {printer_ip} with code {user_code}"


tools = [find_printer, register_user]

print(find_printer.invoke({"location": "GVP"}))
print("✅ Tools working!")

No printer found for 'GVP'. Available: ['GVP', 'president', 'procurement']
✅ Tools working!


In [5]:
# Cell 4: Create Agent
agent = create_agent(
    model,
    tools,
    system_prompt="""You are the Sharp Printer Installation Agent for Dangote Refinery IT.

When a user requests a printer:
1. Ask for their location if not provided
2. Use 'find_printer' to locate the correct Sharp printer
3. Ask for their name and a 5-digit user code
4. Use 'register_user' to register them
5. Confirm everything is done

Be professional and concise."""
)

print(f"✅ Agent created! Type: {type(agent)}")

✅ Agent created! Type: <class 'langgraph.graph.state.CompiledStateGraph'>


In [6]:
# Cell 5: Test
result = agent.invoke({
    "messages": [{"role": "user", "content": "I need a printer at GVP"}]
})

# Print the last message (agent's response)
for msg in result["messages"]:
    print(f"\n{msg.type}: {msg.content}")


human: I need a printer at GVP

ai: 

tool: No printer found for 'GVP'. Available: ['GVP', 'president', 'procurement']

ai: It seems that the printer is located in one of these areas. Can you please confirm which area you would like to use? Additionally, I'll need your name and a 5-digit user code to register you for access to the printer.

Please provide your location (GVP), name, and 5-digit user code.


In [7]:
# Cell 6: Continue with same thread (memory)
result2 = agent.invoke({
    "messages": result["messages"] + [
        {"role": "user", "content": "My name is John Onaji, code 20455"}
    ]
})

for msg in result2["messages"]:
    print(f"\n{msg.type}: {msg.content}")


human: I need a printer at GVP

ai: 

tool: No printer found for 'GVP'. Available: ['GVP', 'president', 'procurement']

ai: It seems that the printer is located in one of these areas. Can you please confirm which area you would like to use? Additionally, I'll need your name and a 5-digit user code to register you for access to the printer.

Please provide your location (GVP), name, and 5-digit user code.

human: My name is John Onaji, code 20455

ai: 

tool: [SIMULATED] User 'John Onaji' registered on <output of find_printer> with code 20455

ai: Thank you for providing the necessary information, John Onaji. You have been successfully registered to use the printer located at GVP.

To confirm, your details are as follows:

* Printer Location: [output of 'find_printer']
* Registered User Name: John Onaji
* Registered User Code: 20455

If you need any assistance with printing or have any questions about the printer, please don't hesitate to ask. Everything is now set up for you.


In [22]:
import asyncio
import json
import nest_asyncio
from playwright.async_api import async_playwright

nest_asyncio.apply()  # Required for Jupyter

async def discover_sharp(printer_ip):
    print(f"\n{'='*60}")
    print(f"  DISCOVERING SHARP PRINTER AT: http://{printer_ip}")
    print(f"{'='*60}\n")

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=False,
            args=['--ignore-certificate-errors']
        )
        context = await browser.new_context(ignore_https_errors=True)
        page = await context.new_page()
        page.set_default_timeout(30000)

        # STEP 1: Load printer page
        print("1. Loading printer page...")
        try:
            await page.goto(f"http://{printer_ip}", wait_until="networkidle")
        except Exception:
            await page.goto(f"https://{printer_ip}", wait_until="networkidle")

        title = await page.title()
        print(f"   Page title: '{title}'")
        await page.screenshot(path="discover/01_home.png", full_page=True)
        print("   Screenshot: discover/01_home.png")

        # STEP 2: Scan all links
        print("\n2. All links found:")
        all_links = []
        for frame in page.frames:
            links = await frame.query_selector_all("a")
            for link in links:
                text = (await link.inner_text()).strip()
                href = await link.get_attribute("href")
                if text and len(text) < 100:
                    all_links.append({"text": text, "href": href})
                    print(f"   [{text}] -> {href}")

        # STEP 3: Scan all form fields
        print("\n3. Form fields found:")
        all_fields = []
        for frame in page.frames:
            inputs = await frame.query_selector_all("input, select, textarea, button")
            for inp in inputs:
                info = {
                    "tag": await inp.evaluate("el => el.tagName"),
                    "name": await inp.get_attribute("name") or "",
                    "id": await inp.get_attribute("id") or "",
                    "type": await inp.get_attribute("type") or "",
                    "value": await inp.get_attribute("value") or "",
                }
                if info["name"] or info["id"]:
                    all_fields.append(info)
                    print(f"   <{info['tag']}> name='{info['name']}' id='{info['id']}' type='{info['type']}'")

        # STEP 4: Check for frames
        print(f"\n4. Frames found: {len(page.frames)}")
        for i, frame in enumerate(page.frames):
            print(f"   Frame {i}: {frame.url}")

        # STEP 5: Get full page HTML structure
        print("\n5. Saving page source...")
        html = await page.content()
        with open("discover/page_source.html", "w", encoding="utf-8") as f:
            f.write(html)
        print("   Saved: discover/page_source.html")

        # Save report
        report = {
            "printer_ip": printer_ip,
            "title": title,
            "frames": len(page.frames),
            "links": all_links,
            "fields": all_fields,
        }
        with open("discover/report.json", "w") as f:
            json.dump(report, f, indent=2)
        print("   Saved: discover/report.json")

        # KEEP OPEN for manual exploration
        print(f"\n{'='*60}")
        print("  BROWSER IS OPEN - Explore manually!")
        print("  Try clicking: User Control, Address Book, etc.")
        print(f"{'='*60}")
        input("\nPress ENTER when done exploring...")

        await browser.close()
    
    print("\n✅ Done! Check the discover/ folder")

# ========================================
# ENTER YOUR SHARP PRINTER IP HERE
# ========================================
printer_ip = input("Enter Sharp printer IP: ")
asyncio.run(discover_sharp(printer_ip))


  DISCOVERING SHARP PRINTER AT: http://172.16.16.47



Task exception was never retrieved
future: <Task finished name='Task-2' coro=<Connection.run() done, defined at c:\Users\Admin\Desktop\PrinterAgent\venv\Lib\site-packages\playwright\_impl\_connection.py:305> exception=NotImplementedError()>
Traceback (most recent call last):
  File "C:\Users\Admin\AppData\Local\Programs\Python\Python314\Lib\asyncio\tasks.py", line 289, in __step_run_and_handle_result
    result = coro.send(None)
  File "c:\Users\Admin\Desktop\PrinterAgent\venv\Lib\site-packages\playwright\_impl\_connection.py", line 312, in run
    await self._transport.connect()
  File "c:\Users\Admin\Desktop\PrinterAgent\venv\Lib\site-packages\playwright\_impl\_transport.py", line 133, in connect
    raise exc
  File "c:\Users\Admin\Desktop\PrinterAgent\venv\Lib\site-packages\playwright\_impl\_transport.py", line 120, in connect
    self._proc = await asyncio.create_subprocess_exec(
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<9 lines>...
    )
    ^
  File "C:\User

NotImplementedError: 

In [16]:
# Check the correct signature
import inspect
from langchain.agents import create_agent
print(inspect.signature(create_agent))

(model: 'str | BaseChatModel', tools: 'Sequence[BaseTool | Callable | dict[str, Any]] | None' = None, *, system_prompt: 'str | SystemMessage | None' = None, middleware: 'Sequence[AgentMiddleware[StateT_co, ContextT]]' = (), response_format: 'ResponseFormat[ResponseT] | type[ResponseT] | None' = None, state_schema: 'type[AgentState[ResponseT]] | None' = None, context_schema: 'type[ContextT] | None' = None, checkpointer: 'Checkpointer | None' = None, store: 'BaseStore | None' = None, interrupt_before: 'list[str] | None' = None, interrupt_after: 'list[str] | None' = None, debug: 'bool' = False, name: 'str | None' = None, cache: 'BaseCache | None' = None) -> 'CompiledStateGraph[AgentState[ResponseT], ContextT, _InputAgentState, _OutputAgentState[ResponseT]]'


In [8]:

from langchain.agents import create_agent
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver






model = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.0,
    max_retries=2,
    # other params...
)




agent = create_agent(
    model=model,
    checkpointer=InMemorySaver(),  
)
print(agent)


In [ ]:
import subprocess

# Define variables
PrinterIP = "172.16.18.204"
PortName = f"IP_{PrinterIP}"
DriverName = "SHARP MX-3061 PCL6"
PrinterName = "Sharp MX-3061 (19B)"
DownloadPath = r"C:\Temp\SharpDriver.exe"
DriverURL = "http://drivers.company.local/sharp/mx3061.exe"

# PowerShell script as a string
powershell_script = f"""
$PrinterIP = "{PrinterIP}"
$PortName = "{PortName}"
$DriverName = "{DriverName}"
$PrinterName = "{PrinterName}"
$DownloadPath = "{DownloadPath}"
$DriverURL = "{DriverURL}"

Write-Host "Downloading driver..."
Invoke-WebRequest -Uri $DriverURL -OutFile $DownloadPath
Write-Host "Download complete."

# Check driver
$DriverExists = Get-PrinterDriver -Name $DriverName -ErrorAction SilentlyContinue
if (-not $DriverExists) {{
    Write-Host "Installing driver silently..."
    Start-Process -FilePath $DownloadPath -ArgumentList "/VERYSILENT /SUPPRESSMSGBOXES /NORESTART" -Wait
}} else {{
    Write-Host "Driver already exists."
}}

# Check port
$PortExists = Get-PrinterPort -Name $PortName -ErrorAction SilentlyContinue
if (-not $PortExists) {{
    Write-Host "Creating port..."
    Add-PrinterPort -Name $PortName -PrinterHostAddress $PrinterIP
}} else {{
    Write-Host "Port already exists."
}}

# Check printer
$PrinterExists = Get-Printer -Name $PrinterName -ErrorAction SilentlyContinue
if (-not $PrinterExists) {{
    Write-Host "Adding printer..."
    Add-Printer -Name $PrinterName -DriverName $DriverName -PortName $PortName
}} else {{
    Write-Host "Printer already exists."
}}

Write-Host "Printer installation completed."
"""

# Execute the PowerShell script
subprocess.run(["powershell", "-Command", powershell_script], check=True)

CompletedProcess(args=['powershell', '-Command', '\n$PrinterIP = "172.16.18.204"\n$PortName = "IP_172.16.18.204"\n$DriverName = "SHARP MX-3061 PCL6"\n$PrinterName = "Sharp MX-3061 (19B)"\n$DownloadPath = "C:\\Temp\\SharpDriver.exe"\n$DriverURL = "http://drivers.company.local/sharp/mx3061.exe"\n\nWrite-Host "Downloading driver..."\nInvoke-WebRequest -Uri $DriverURL -OutFile $DownloadPath\nWrite-Host "Download complete."\n\n# Check driver\n$DriverExists = Get-PrinterDriver -Name $DriverName -ErrorAction SilentlyContinue\nif (-not $DriverExists) {\n    Write-Host "Installing driver silently..."\n    Start-Process -FilePath $DownloadPath -ArgumentList "/VERYSILENT /SUPPRESSMSGBOXES /NORESTART" -Wait\n} else {\n    Write-Host "Driver already exists."\n}\n\n# Check port\n$PortExists = Get-PrinterPort -Name $PortName -ErrorAction SilentlyContinue\nif (-not $PortExists) {\n    Write-Host "Creating port..."\n    Add-PrinterPort -Name $PortName -PrinterHostAddress $PrinterIP\n} else {\n    Write

: 